In [2]:
%pip install pandas sqlalchemy pyodbc

Note: you may need to restart the kernel to use updated packages.


In [39]:
!pip install openpyxl


   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]



In [2]:
import pandas as pd
import pyodbc
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

In [3]:
server = r"localhost\SQLEXPRESS01"
database = "EcommerceCustomerAnalytics"

connection_url = URL.create(
    "mssql+pyodbc",
    query={
        "odbc_connect": (
            "DRIVER={ODBC Driver 18 for SQL Server};"
            f"SERVER={server};"
            f"DATABASE={database};"
            "Trusted_Connection=yes;"
            "TrustServerCertificate=yes;"
        )
    }
)

engine = create_engine(connection_url)

with engine.connect() as conn:
    print("Connected to SQL Server successfully!")

Connected to SQL Server successfully!


In [4]:
from sqlalchemy import text

conn = engine.connect()

print("Connection established successfully!")

Connection established successfully!


In [5]:
result = conn.execute(text("SELECT DB_NAME() AS DatabaseName"))

print(result.fetchone())

('EcommerceCustomerAnalytics',)


storing the dataset in df

In [6]:
query = """
SELECT *
FROM dbo.CleanedRetailData
"""

In [7]:
df = pd.read_sql_query(query, engine)

In [8]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer_id,Country,SourceSheet
0,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2010-2011
1,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom,2010-2011
2,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2010-2011
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,2010-2011
4,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom,2010-2011


In [9]:
df.shape

(1033036, 9)

In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1033036 entries, 0 to 1033035
Data columns (total 9 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1033036 non-null  str           
 1   StockCode    1033036 non-null  str           
 2   Description  1028761 non-null  str           
 3   Quantity     1033036 non-null  int64         
 4   InvoiceDate  1033036 non-null  datetime64[us]
 5   Price        1033036 non-null  float64       
 6   Customer_id  797885 non-null   float64       
 7   Country      1033036 non-null  str           
 8   SourceSheet  1033036 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), str(5)
memory usage: 70.9 MB


In [11]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

In [30]:
#calculate and filter recency, frequency, monetary

rfm_data = df[(df['Customer_id'].notna()) & (df['Quantity']>0)].copy()
snapshot_date =pd.to_datetime('2011-12-10')

rfm = rfm_data.groupby('Customer_id').agg(
    Recency = ('InvoiceDate', lambda x: (snapshot_date -x.max()).days),
    Frequency = ('Invoice','nunique'),
    Monetary = ('Quantity', lambda x : (x*rfm_data.loc[x.index,'Price']).sum())
).reset_index()

In [31]:
rfm_data

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer_id,Country,SourceSheet
0,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2010-2011
1,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom,2010-2011
2,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2010-2011
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,2010-2011
4,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom,2010-2011
...,...,...,...,...,...,...,...,...,...
1013921,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France,2009-2010
1013922,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France,2009-2010
1013923,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France,2009-2010
1013924,581587,23256,CHILDRENS CUTLERY SPACEBOY,4,2011-12-09 12:50:00,4.15,12680.0,France,2009-2010


In [32]:
rfm.isna().sum()

Customer_id    0
Recency        0
Frequency      0
Monetary       0
dtype: int64

In [33]:
rfm['r_score'] = pd.qcut(rfm['Recency'],5, labels = [5,4,3,2,1]).astype(int)
rfm['f_score'] = pd.qcut(rfm['Frequency'].rank(method='first'),5,labels = [1,2,3,4,5]).astype(int)
rfm['m_score'] = pd.qcut(rfm['Monetary'],5, labels = [1,2,3,4,5]).astype(int)

def segment_customer(row):
    if row['r_score']>=4 and row['f_score'] >= 4 and row['m_score']>=4:
        return 'Champions'
    elif row['r_score']>=3 and row['f_score'] >= 3:
        return 'Loyal'
    elif row['r_score']>=2 and row['f_score'] >= 2:
        return 'At Risk'
    elif row['r_score']>=4 and row['f_score'] <= 2:
        return 'New'
    elif row['r_score']>=1 and row['f_score'] >= 1:
        return 'Needs Attention'
    else:
        return 'Lost'

rfm['Segment'] = rfm.apply(segment_customer, axis=1)
rfm['IsChurned'] = rfm['Recency']>90

In [34]:
rfm

,Customer_id,Recency,Frequency,Monetary,r_score,f_score,m_score,Segment,IsChurned
0,12346.0,325,12,77556.46,2,5,5,At Risk,True
1,12347.0,2,8,4921.53,5,4,5,Champions,False
2,12348.0,75,5,2019.40,3,4,4,Loyal,False
3,12349.0,18,4,4428.69,5,3,5,Loyal,False
4,12350.0,310,1,334.40,2,1,2,Needs Attention,True
...,...,...,...,...,...,...,...,...,...
5876,18283.0,3,22,2664.90,5,5,4,Champions,False
5877,18284.0,431,1,461.68,1,2,2,Needs Attention,True
5878,18285.0,660,1,427.00,1,2,2,Needs Attention,True
5879,18286.0,476,2,1296.43,1,3,4,Needs Attention,True


In [35]:
rfm.isna().sum()

Customer_id    0
Recency        0
Frequency      0
Monetary       0
r_score        0
f_score        0
m_score        0
Segment        0
IsChurned      0
dtype: int64

In [17]:
# #merging RFM results and exporting

# df =  df[(df['Customer_id'].notna()) & (df['Quantity'])>0].copy()
# df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M').dt.to_timestamp()
# df['Revenue'] = df['Quantity']*df['Price']

# df_final = df.merge(rfm[['Customer_id','Recency','Frequency','Monetary','r_score','f_score','m_score','Segment','IsChurned']],
#                    on = 'Customer_id', how = 'left')

# df_final.to_excel('ECommerceDashboardData.xlsx',index = False)

In [36]:
# 1. Ensure correct datetime format
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# 2. Filter valid transactions
df_clean = df[df['Customer_id'].notna() & (df['Quantity'] > 0)].copy()

# 3. Feature engineering
df_clean['InvoiceMonth'] = df_clean['InvoiceDate'].dt.to_period('M').dt.to_timestamp()
df_clean['Revenue'] = df_clean['Quantity'] * df_clean['Price']

# 4. Merge transaction-level data with customer RFM attributes
rfm_columns = [
    'Customer_id', 'Recency', 'Frequency', 'Monetary',
    'r_score', 'f_score', 'm_score', 'Segment', 'IsChurned'
]

df_final = df_clean.merge(rfm[rfm_columns], on='Customer_id', how='left')

# 5. Export to Excel (or CSV for large datasets)
# openpyxl engine is standard for modern .xlsx and .csv
df_final.to_csv('ECommerceDashboardData.csv', index=False)


In [38]:
# 1. Re-initialize the engine with fast_executemany enabled for bulk writing
engine = create_engine(connection_url, fast_executemany=True)

# 2. Write df_final into SQL Server
table_name = "ECommerceDashboardData"

df_final.to_sql(
    name=table_name,
    con=engine,
    if_exists="replace",  # Use 'replace' to create/overwrite, or 'append' to insert into an existing table
    index=False,
    chunksize=10000       # Writes in batches to manage memory efficiently
)

print(f"Data successfully saved to table '{table_name}' in database '{database}'!")

Data successfully saved to table 'ECommerceDashboardData' in database 'EcommerceCustomerAnalytics'!


In [20]:
conn.close()